# Random Forest

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score


features = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms']

train = pd.read_csv('data/splits/train.csv')
val = pd.read_csv('data/splits/val.csv')
test = pd.read_csv('data/splits/test.csv')

X_train, y_train = train[features], train['genre_group']
X_val, y_val = val[features], val['genre_group']
X_test, y_test = test[features], test['genre_group']

print(len(train), len(val), len(test))  # should be roughly 80 / 10 / 10

68633 8579 8580


### using balanced_subsample for class weight

In [2]:
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=3, class_weight='balanced_subsample', random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_val)
print(classification_report(y_val, rf_pred))

                      precision    recall  f1-score   support

chill_jazz_classical       0.60      0.69      0.64      1038
             country       0.42      0.56      0.48       293
    electronic_dance       0.67      0.72      0.70      1932
          hiphop_rnb       0.42      0.28      0.33       385
          indie_folk       0.38      0.36      0.37       251
               latin       0.48      0.60      0.53       917
                 pop       0.46      0.39      0.42      1096
              reggae       0.31      0.30      0.30       194
          rock_metal       0.62      0.58      0.60      1684
               world       0.38      0.28      0.32       789

            accuracy                           0.55      8579
           macro avg       0.47      0.48      0.47      8579
        weighted avg       0.54      0.55      0.54      8579



### using unbalanced for class weight

In [3]:
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=3, class_weight=None, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_val)
print(classification_report(y_val, rf_pred))

                      precision    recall  f1-score   support

chill_jazz_classical       0.60      0.70      0.64      1038
             country       0.59      0.42      0.49       293
    electronic_dance       0.61      0.79      0.69      1932
          hiphop_rnb       0.74      0.18      0.30       385
          indie_folk       0.63      0.25      0.35       251
               latin       0.49      0.58      0.53       917
                 pop       0.45      0.40      0.42      1096
              reggae       0.76      0.16      0.26       194
          rock_metal       0.57      0.65      0.61      1684
               world       0.45      0.24      0.32       789

            accuracy                           0.56      8579
           macro avg       0.59      0.44      0.46      8579
        weighted avg       0.56      0.56      0.54      8579



# testing/tweaking parameters

#### changing min_samples_leaf and max_features

Unbalanced ----------

In [4]:
best_score = 0
best_params = None

for leaf in [1, 3, 5, 10]:
    for mf in ['sqrt', 0.5, 1.0]:
        rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=leaf, max_features=mf, n_jobs=-1, random_state=42)
        rf.fit(X_train, y_train)
        score = f1_score(y_val, rf.predict(X_val), average='macro')
        print(f"min_samples_leaf={leaf:<3} max_features={mf!s:<5} val macro-F1 = {score:.3f}")

        if score > best_score:
            best_score = score
            best_params = {'min_samples_leaf': leaf, 'max_features': mf}

print("\nBest:", best_params, f"-> {best_score:.3f}")

min_samples_leaf=1   max_features=sqrt  val macro-F1 = 0.469
min_samples_leaf=1   max_features=0.5   val macro-F1 = 0.468
min_samples_leaf=1   max_features=1.0   val macro-F1 = 0.470
min_samples_leaf=3   max_features=sqrt  val macro-F1 = 0.461
min_samples_leaf=3   max_features=0.5   val macro-F1 = 0.462
min_samples_leaf=3   max_features=1.0   val macro-F1 = 0.459
min_samples_leaf=5   max_features=sqrt  val macro-F1 = 0.445
min_samples_leaf=5   max_features=0.5   val macro-F1 = 0.449
min_samples_leaf=5   max_features=1.0   val macro-F1 = 0.446
min_samples_leaf=10  max_features=sqrt  val macro-F1 = 0.410
min_samples_leaf=10  max_features=0.5   val macro-F1 = 0.417
min_samples_leaf=10  max_features=1.0   val macro-F1 = 0.414

Best: {'min_samples_leaf': 1, 'max_features': 1.0} -> 0.470


Balanced ----------

In [5]:
best_score = 0
best_params = None

for leaf in [1, 3, 5, 10]:
    for mf in ['sqrt', 0.5, 1.0]:
        rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=leaf, max_features=mf, class_weight='balanced_subsample', n_jobs=-1, random_state=42)
        rf.fit(X_train, y_train)
        score = f1_score(y_val, rf.predict(X_val), average='macro')
        print(f"min_samples_leaf={leaf:<3} max_features={mf!s:<5} val macro-F1 = {score:.3f}")

        if score > best_score:
            best_score = score
            best_params = {'min_samples_leaf': leaf, 'max_features': mf}

print("\nBest:", best_params, f"-> {best_score:.3f}")

min_samples_leaf=1   max_features=sqrt  val macro-F1 = 0.470
min_samples_leaf=1   max_features=0.5   val macro-F1 = 0.473
min_samples_leaf=1   max_features=1.0   val macro-F1 = 0.468
min_samples_leaf=3   max_features=sqrt  val macro-F1 = 0.471
min_samples_leaf=3   max_features=0.5   val macro-F1 = 0.473
min_samples_leaf=3   max_features=1.0   val macro-F1 = 0.465
min_samples_leaf=5   max_features=sqrt  val macro-F1 = 0.456
min_samples_leaf=5   max_features=0.5   val macro-F1 = 0.461
min_samples_leaf=5   max_features=1.0   val macro-F1 = 0.457
min_samples_leaf=10  max_features=sqrt  val macro-F1 = 0.432
min_samples_leaf=10  max_features=0.5   val macro-F1 = 0.434
min_samples_leaf=10  max_features=1.0   val macro-F1 = 0.432

Best: {'min_samples_leaf': 3, 'max_features': 0.5} -> 0.473


In [6]:
best_score = 0
best_est = 0

for est in [100, 200, 300, 400, 500]:
    rf = RandomForestClassifier(n_estimators=est, min_samples_leaf=best_params['min_samples_leaf'], max_features=best_params['max_features'], class_weight='balanced_subsample', n_jobs=-1, random_state=42)
    rf.fit(X_train, y_train)
    score = f1_score(y_val, rf.predict(X_val), average='macro')
    print(f"n_estimators={est:<3} val macro-F1 = {score:.3f}")

    if score > best_score:
        best_score = score
        best_est = est

print("\nBest:", est, f"-> {best_score:.3f}")

n_estimators=100 val macro-F1 = 0.470
n_estimators=200 val macro-F1 = 0.473
n_estimators=300 val macro-F1 = 0.468
n_estimators=400 val macro-F1 = 0.470
n_estimators=500 val macro-F1 = 0.473

Best: 500 -> 0.473


# Results

In [7]:
rf = RandomForestClassifier(n_estimators=500, min_samples_leaf=3, max_features=0.5, class_weight='balanced_subsample',  n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_val)
print(classification_report(y_val, rf_pred))

                      precision    recall  f1-score   support

chill_jazz_classical       0.62      0.69      0.65      1038
             country       0.43      0.55      0.48       293
    electronic_dance       0.68      0.73      0.71      1932
          hiphop_rnb       0.43      0.27      0.33       385
          indie_folk       0.35      0.34      0.34       251
               latin       0.48      0.60      0.53       917
                 pop       0.46      0.41      0.43      1096
              reggae       0.32      0.30      0.31       194
          rock_metal       0.62      0.59      0.60      1684
               world       0.39      0.29      0.34       789

            accuracy                           0.55      8579
           macro avg       0.48      0.48      0.47      8579
        weighted avg       0.55      0.55      0.55      8579



In [8]:
print(classification_report(y_test, rf.predict(X_test)))

                      precision    recall  f1-score   support

chill_jazz_classical       0.63      0.70      0.66      1038
             country       0.45      0.58      0.51       292
    electronic_dance       0.67      0.72      0.69      1932
          hiphop_rnb       0.45      0.31      0.37       385
          indie_folk       0.37      0.32      0.35       250
               latin       0.50      0.66      0.57       918
                 pop       0.43      0.38      0.41      1097
              reggae       0.31      0.29      0.30       194
          rock_metal       0.65      0.59      0.62      1685
               world       0.40      0.30      0.34       789

            accuracy                           0.56      8580
           macro avg       0.49      0.49      0.48      8580
        weighted avg       0.55      0.56      0.55      8580

